In [ ]:
import os

print(os.cpu_count(), len(os.sched_getaffinity(0)))
n = len(os.sched_getaffinity(0))
os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["XLA_FLAGS"] = (
    "--xla_cpu_multi_thread_eigen=true "
    f"intra_op_parallelism_threads={n} "
    f"--xla_force_host_platform_device_count={n}"  # optional, see below
)

import jax
import master.device as device
from master.obj import *
from master.estimator_bounds import *
import numpy as np
from master import *
from pty_defs import *

jax.config.update("jax_enable_x64", True)
cdtype = jnp.array(1j).dtype
jax.devices()
device_id = 0
device.setup_device(device_id=device_id)

In [ ]:
base_path = "/data/multispot_pty/pty-64-jitter-full/"
# base_path = "/data/multispot_pty/pty-64-jitter-10k/"
loader = Loader(base_path)

fig_path = "../docs/thesis/figures"
SAVE_FIGS = True

In [ ]:
from master.digest import make_array_digest
from master.probes import construct_probe_fields, construct_probe_inf_probe_approx
from master.spotgen import create_multispot_array_product_jittered

pixel_intensity = 1e3
meas_id = f"CDI-intensity-{pixel_intensity:.3g}"
est_id = "ORACLE-second-order-ml"
est_label = "ORACLE+ML"


x_true, d, probes, spectral_support, _, measurement_params, measurement_hash_meas = (
    loader.get_measurement_with_probes(meas_id)
)
del x_true, d, probes, spectral_support, _

P_spectral_support = lambda x: ifft2_shifted(fft2_shifted(x) * spectral_support)

meas_id = f"CDI-intensity-{pixel_intensity:.3g}"
x_est, simulation_params, lls, x_est_raw, config_hash, measurement_hash_est = (
    loader.get_estimate_with_x_est_raw(meas_id, est_id)
)
del x_est, lls, x_est_raw
# Check that measurement corresponds to reconstruction
assert measurement_hash_est == measurement_hash_meas
sim_params = simulation_params["simulation"]
probe_dict = simulation_params["probe"]
solver_params = simulation_params["solver"]
ml_params = solver_params["ml_stage"]

N_crop = sim_params["N_crop"]
N_pad = sim_params["N_pad"]
N_photons = sim_params["N_photons"]
N_samples = sim_params["N_samples"]
M = sim_params["M"]

N_RES = 4286
N_old = N_crop + 2 * N_pad
N_pad = (N_RES - N_crop) // 2

frac_scale = 0.5
ratio = N_old / (frac_scale * N_RES)
print(N_pad, N_crop)

x_true_fake = np.zeros((N_RES, N_RES))

NA_det = sim_params["NA_det"]

N_iter_ML = ml_params["N_iter_ML"]

probe_params = ProbeParams.from_dict(probe_dict)
print(probe_params)
pty_sim = PtySim(N_crop, N_pad, NA_det, x_true=x_true_fake, be=np)
# A, A_dagger, F_pupil, pupil_support = pty_sim.get_ops_and_pupil()

probe_params.NA_spot *= ratio
probe_params.frac_x *= frac_scale
probe_params.frac_y *= frac_scale
base_probes, N_photons, N_coeff = pty_sim.create_base_probe(
    probe_params,
    pixel_intensity,
    f_probe=create_multispot_array_product_jittered,
    use_shift=True,
    r_frac_jitter=0.1,
)
base_probe = base_probes[0]
# R_probes = pty_sim.get_R_probes(M,0.5,probe_params)

# x_true.shape, d.shape, x_est.shape, simulation_params["simulation"], N_d_crop

In [ ]:
# cmaps = ["viridis","plasma","inferno","magma","cividis","gray","Greys","twilight","twilight_shifted"]
shift = 600


def crop(img, w=2857):
    x0 = (img.shape[1] - w) // 2
    return img[:, x0 : x0 + w]  # centered


def make_cover(img):
    return crop(np.roll(img, shift, axis=0))


cmaps = ["Greys"]
for cmap in cmaps:
    plt.imsave(
        f"{fig_path}/multispot_{cmap}.png",
        make_cover(np.real(base_probes[0])),
        cmap=cmap,
        origin="upper",
    )
    plt.imsave(
        f"{fig_path}/multispot_{cmap}_inv.png",
        make_cover(-np.real(base_probes[0])),
        cmap=cmap,
        origin="upper",
    )

In [ ]:
from master.noise import *

field = base_probes[0]
norm_field = field / jnp.sqrt(jnp.sum(jnp.abs(field) ** 2))
N_scale = 1e5
_, noisy_field = sample_from_mu(N_scale * jnp.abs(norm_field) ** 2)
noisy_field = noisy_field[0]
plot_intensity_phase(noisy_field)

In [ ]:
for cmap in cmaps:
    plt.imsave(
        f"{fig_path}/multispot_{cmap}_noisy_int_{N_scale:.3g}.png",
        make_cover(np.abs(noisy_field) ** 2),
        cmap=cmap,
        origin="upper",
    )
for cmap in cmaps:
    plt.imsave(
        f"{fig_path}/multispot_{cmap}_noisy_int_inv_{N_scale:.3g}.png",
        make_cover(-np.abs(noisy_field) ** 2),
        cmap=cmap,
        origin="upper",
    )